In [1]:
import csv

import io

def parse_plakoro_csv(data_source, is_file_path=False):
    result = {}
    current_set = None
    
    # Open file or read from string
    if is_file_path:
        f = open(data_source, mode='r', encoding='utf-8')
    else:
        f = io.StringIO(data_source.strip())
        
    reader = csv.reader(f)
    next(reader)  # Skip the header row
    
    for row in reader:
        # Pad row to ensure it has at least 6 columns to prevent IndexError
        row += [''] * (6 - len(row))
        
        set_name = row[0].strip()
        mix_1 = row[1].strip()
        mix_2 = row[2].strip()
        mix_count = row[3].strip()
        single_element = row[4].strip()
        single_count = row[5].strip()
        
        # If a new set name is found, switch the active key
        if set_name:
            current_set = set_name
            result[current_set] = []
            
        if not current_set:
            continue
            
        # 1. Process Mixed Faces (Columns 1, 2, 3)
        if mix_1 and mix_2 and mix_count:
            result[current_set].append({
                "faces": [mix_1, mix_2],
                "count": int(mix_count)
            })
            
        # 2. Process Single Faces (Columns 4, 5)
        if single_element and single_count:
            result[current_set].append({
                "faces": [single_element],
                "count": int(single_count)
            })
            
    if is_file_path:
        f.close()
        
    return result

# --- How to run it ---

# Option A: From the string data above
csv_data = open("plakoro sets - dice faces count.csv", mode='r', encoding='utf-8').read()
output = parse_plakoro_csv(csv_data)

# Option B: From your uploaded file (uncomment below to use)
# output = parse_plakoro_csv("plakoro - dice faces count.csv", is_file_path=True)

# Pretty print the output to verify
import json
print(json.dumps(output, ensure_ascii=False, indent=4))

{
    "Plakoro Starter Set Bulbasaur 01": [
        {
            "faces": [
                "หญ้า",
                "หญ้า"
            ],
            "count": 5
        },
        {
            "faces": [
                "หญ้า"
            ],
            "count": 3
        },
        {
            "faces": [
                "หญ้า",
                "ความมืด"
            ],
            "count": 2
        },
        {
            "faces": [
                "บิน"
            ],
            "count": 1
        },
        {
            "faces": [
                "ความมืด",
                "ความมืด"
            ],
            "count": 1
        },
        {
            "faces": [
                "น้ำ"
            ],
            "count": 1
        },
        {
            "faces": [
                "ต่อสู้",
                "ต่อสู้"
            ],
            "count": 1
        },
        {
            "faces": [
                "ไฟ"
            ],
            "count": 1
        },
        {
 

In [2]:
import random

import pandas as pd
import ipywidgets as widgets
from collections import Counter
from IPython.display import display

set_names = sorted(output.keys())
set_selector = widgets.Dropdown(
    options=[(name, name) for name in set_names],
    description='Set:',
    layout=widgets.Layout(width='420px')
)
preview_output = widgets.Output()
plakoro_state = {}


def format_faces(faces):
    return '/'.join(faces)


def build_face_pool(selected_set):
    pool = []
    for group_index, item in enumerate(output[selected_set]):
        label = format_faces(item['faces'])
        for occurrence in range(item['count']):
            token = f'{group_index}:{occurrence}:{label}'
            pool.append({'label': label, 'token': token})
    return pool


def render_preview(change=None):
    with preview_output:
        preview_output.clear_output(wait=True)

        selected_set = set_selector.value
        face_pool = build_face_pool(selected_set)
        label_by_token = {entry['token']: entry['label'] for entry in face_pool}
        total_counts = Counter(entry['label'] for entry in face_pool)

        table_output = widgets.Output()
        counts_output = widgets.Output()
        state = {'updating': False}

        slot_dropdowns = []
        die_boxes = []

        for die_number in range(1, 4):
            die_dropdowns = []
            for face_number in range(1, 7):
                dropdown = widgets.Dropdown(
                    options=[('—', None)],
                    description=f'Face {face_number}:',
                    layout=widgets.Layout(width='220px')
                )
                die_dropdowns.append(dropdown)
                slot_dropdowns.append(dropdown)

            die_boxes.append(
                widgets.VBox(
                    [
                        widgets.HTML(f'<h4 style="margin:0 0 6px 0;">Die {die_number}</h4>'),
                        *die_dropdowns,
                    ],
                    layout=widgets.Layout(padding='8px 12px', border='1px solid #ddd')
                )
            )

        plakoro_state['selected_set'] = selected_set
        plakoro_state['face_pool'] = face_pool
        plakoro_state['label_by_token'] = label_by_token
        plakoro_state['total_counts'] = total_counts
        plakoro_state['slot_dropdowns'] = slot_dropdowns

        def draw_tables():
            selected_labels_by_die = [
                [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[0:6]],
                [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[6:12]],
                [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[12:18]],
            ]

            with table_output:
                table_output.clear_output(wait=True)
                face_table = pd.DataFrame(
                    {
                        'Die 1': selected_labels_by_die[0],
                        'Die 2': selected_labels_by_die[1],
                        'Die 3': selected_labels_by_die[2],
                    },
                    index=[f'Face {index}' for index in range(1, 7)]
                )
                display(face_table)

            with counts_output:
                counts_output.clear_output(wait=True)
                used_counts = Counter(label_by_token[dropdown.value] for dropdown in slot_dropdowns if dropdown.value is not None)
                remaining_rows = []
                for label, total in total_counts.items():
                    used = used_counts.get(label, 0)
                    remaining_rows.append({
                        'Face': label,
                        'Used': used,
                        'Limit': total,
                        'Remaining': total - used,
                    })
                display(pd.DataFrame(remaining_rows))

            # if 'plakoro_update_probability_summary' in globals():
            #     plakoro_update_probability_summary()

        def refresh_options(_change=None):
            if state['updating']:
                return

            state['updating'] = True
            try:
                selected_tokens = {dropdown.value for dropdown in slot_dropdowns if dropdown.value is not None}

                for dropdown in slot_dropdowns:
                    current_value = dropdown.value
                    other_selected_tokens = selected_tokens - ({current_value} if current_value is not None else set())
                    allowed_entries = [entry for entry in face_pool if entry['token'] not in other_selected_tokens]
                    options = [('—', None)] + [(entry['label'], entry['token']) for entry in allowed_entries]
                    dropdown.options = options
                    available_values = {value for _, value in options}
                    dropdown.value = current_value if current_value in available_values else None
            finally:
                state['updating'] = False

            draw_tables()

        def randomize_selection(_button=None):
            if not face_pool:
                return

            state['updating'] = True
            try:
                full_options = [('—', None)] + [(entry['label'], entry['token']) for entry in face_pool]
                shuffled_tokens = [entry['token'] for entry in face_pool]
                random.shuffle(shuffled_tokens)
                selected_tokens = shuffled_tokens[:len(slot_dropdowns)]

                for dropdown in slot_dropdowns:
                    dropdown.options = full_options
                    dropdown.value = None

                for dropdown, token in zip(slot_dropdowns, selected_tokens):
                    dropdown.value = token
            finally:
                state['updating'] = False

            refresh_options()

        random_button = widgets.Button(
            description='Random face selection',
            button_style='primary',
            icon='shuffle',
            layout=widgets.Layout(width='220px')
        )
        random_button.on_click(randomize_selection)

        for dropdown in slot_dropdowns:
            dropdown.observe(refresh_options, names='value')

        refresh_options()

        display(
            widgets.VBox(
                [
                    widgets.HTML(
                        '<h3 style="margin:0 0 8px 0;">Choose a Plakoro set</h3>'
                        '<div style="margin:0 0 10px 0;color:#666;">Each face combination can only be selected as many times as its count.</div>'
                    ),
                    widgets.HBox([random_button], layout=widgets.Layout(margin='0 0 6px 0')),
                    widgets.HBox(die_boxes, layout=widgets.Layout(gap='24px', flex_wrap='wrap')),
                    widgets.HTML('<h4 style="margin:12px 0 6px 0;">Current selection</h4>'),
                    table_output,
                    widgets.HTML('<h4 style="margin:12px 0 6px 0;">Remaining counts</h4>'),
                    counts_output,
                ],
                layout=widgets.Layout(gap='12px')
            )
        )


set_selector.observe(render_preview, names='value')
display(widgets.VBox([set_selector, preview_output], layout=widgets.Layout(gap='12px')))
render_preview()

In [5]:
import itertools

import pandas as pd
import ipywidgets as widgets
from collections import defaultdict, Counter
from IPython.display import display

probability_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd", padding="4px", height="300px", overflow="auto"
    )
)


def plakoro_update_probability_summary():
    if "plakoro_state" not in globals() or not plakoro_state.get("slot_dropdowns"):
        return

    slot_dropdowns = plakoro_state["slot_dropdowns"]
    label_by_token = plakoro_state["label_by_token"]

    die_faces = [
        [
            label_by_token[dropdown.value] if dropdown.value is not None else "—"
            for dropdown in slot_dropdowns[0:6]
        ],
        [
            label_by_token[dropdown.value] if dropdown.value is not None else "—"
            for dropdown in slot_dropdowns[6:12]
        ],
        [
            label_by_token[dropdown.value] if dropdown.value is not None else "—"
            for dropdown in slot_dropdowns[12:18]
        ],
    ]

    def expand_label(label):
        return label.split("/")

    grouped_rows = defaultdict(lambda: {"combos": 0})
    total_rolls = 6**3

    for row_1, row_2, row_3 in itertools.product(range(6), repeat=3):
        labels = (die_faces[0][row_1], die_faces[1][row_2], die_faces[2][row_3])
        if "—" in labels:
            continue

        expanded_counts = Counter()
        for label in labels:
            expanded_counts.update(expand_label(label))

        key_parts = [
            f"{face} x{count}" for face, count in sorted(expanded_counts.items())
        ]
        combo_key = " | ".join(key_parts)
        grouped_rows[combo_key]["combos"] += 1

    probability_rows = []
    for combo_key, info in sorted(
        grouped_rows.items(), key=lambda item: item[1]["combos"], reverse=True
    ):
        probability_rows.append(
            {
                "Face Counts": combo_key,
                "Combinations": info["combos"],
                "Probability %": f"{(info['combos'] / total_rolls) * 100:.2f}%",
            }
        )

    with probability_output:
        probability_output.clear_output(wait=True)
        with pd.option_context("display.max_rows", None, "display.min_rows", None):
            if probability_rows:
                display(pd.DataFrame(probability_rows))
            else:
                display(
                    pd.DataFrame(
                        columns=["Face Counts", "Combinations", "Probability %"]
                    )
                )


display(
    widgets.VBox(
        [
            widgets.HTML(
                '<h4 style="margin:12px 0 6px 0;">Roll probability</h4>'
                '<div style="margin:0 0 10px 0;color:#666;">Mixed faces are expanded into their component types, so หญ้า/หญ้า counts as 2 หญ้า in the grouped result.</div>'
            ),
            probability_output,
        ],
        layout=widgets.Layout(gap="12px"),
    )
)
plakoro_update_probability_summary()

In [6]:
import itertools

import pandas as pd
import ipywidgets as widgets
from collections import Counter, defaultdict
from IPython.display import display

exact_probability_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='4px',
        height='320px',
        overflow='auto',
    )
)


def _get_current_die_faces():
    if 'plakoro_state' not in globals() or not plakoro_state.get('slot_dropdowns'):
        return None

    slot_dropdowns = plakoro_state['slot_dropdowns']
    label_by_token = plakoro_state['label_by_token']
    return [
        [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[0:6]],
        [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[6:12]],
        [label_by_token[dropdown.value] if dropdown.value is not None else '—' for dropdown in slot_dropdowns[12:18]],
    ]


def _collect_face_types(die_faces):
    face_types = set()
    for die in die_faces:
        for label in die:
            if label == '—':
                continue
            for face in label.split('/'):
                face_types.add(face)
    return sorted(face_types)


def _build_at_least_probability_rows(selected_face, target_count):
    die_faces = _get_current_die_faces()
    if not die_faces:
        return None, None, None

    grouped_rows = defaultdict(lambda: {'combos': 0})
    total_rolls = 6 ** 3

    for row_1, row_2, row_3 in itertools.product(range(6), repeat=3):
        labels = (die_faces[0][row_1], die_faces[1][row_2], die_faces[2][row_3])
        if '—' in labels:
            continue

        face_count = 0
        expanded_counts = Counter()
        for label in labels:
            parts = label.split('/')
            expanded_counts.update(parts)
            face_count += sum(1 for part in parts if part == selected_face)

        if face_count >= target_count:
            key_parts = [f'{face} x{count}' for face, count in sorted(expanded_counts.items())]
            combo_key = ' | '.join(key_parts)
            grouped_rows[combo_key]['combos'] += 1

    matching_rows = []
    for combo_key, info in sorted(grouped_rows.items(), key=lambda item: item[1]['combos'], reverse=True):
        matching_rows.append({
            'Face Counts': combo_key,
            'Combinations': info['combos'],
            'Probability %': f"{(info['combos'] / total_rolls) * 100:.2f}%",
        })

    total_matching = sum(item['combos'] for item in grouped_rows.values())
    total_probability = (total_matching / total_rolls) * 100
    return matching_rows, total_matching, total_probability


def render_exact_probability_summary(_change=None):
    die_faces = _get_current_die_faces()
    with exact_probability_output:
        exact_probability_output.clear_output(wait=True)

        if not die_faces:
            display(pd.DataFrame(columns=['Face Counts', 'Combinations', 'Probability %']))
            print('Open the selector cell first so the current dice faces are available.')
            return

        face_types = _collect_face_types(die_faces)
        if not face_types:
            display(pd.DataFrame(columns=['Face Counts', 'Combinations', 'Probability %']))
            print('No selectable face types are available in the current selection.')
            return

        face_selector = widgets.Dropdown(
            options=[(face, face) for face in face_types],
            description='Face:',
            layout=widgets.Layout(width='260px')
        )
        count_input = widgets.BoundedIntText(
            value=3,
            min=0,
            max=18,
            description='Count:',
            layout=widgets.Layout(width='180px')
        )
        result_output = widgets.Output()

        def update_result(_inner_change=None):
            selected_face = face_selector.value
            target_count = count_input.value
            rows, total_matching, total_probability = _build_at_least_probability_rows(selected_face, target_count)

            with result_output:
                result_output.clear_output(wait=True)
                print(f'Probability for {selected_face} x {target_count} or more: {total_matching} / 216 = {total_probability:.2f}%')
                if rows:
                    display(pd.DataFrame(rows))
                else:
                    display(pd.DataFrame(columns=['Face Counts', 'Combinations', 'Probability %']))

        face_selector.observe(update_result, names='value')
        count_input.observe(update_result, names='value')

        display(
            widgets.VBox(
                [
                    widgets.HTML(
                        '<h4 style="margin:12px 0 6px 0;">At-least face-count probability</h4>'
                        '<div style="margin:0 0 10px 0;color:#666;">Pick a face type and a count, then the result shows every outcome with that face appearing that many times or more.</div>'
                    ),
                    widgets.HBox([face_selector, count_input], layout=widgets.Layout(gap='12px')),
                    result_output,
                ],
                layout=widgets.Layout(gap='12px')
            )
        )

        update_result()


display(
    widgets.VBox(
        [
            widgets.HTML(
                '<h4 style="margin:12px 0 6px 0;">At-least face-count calculator</h4>'
                '<div style="margin:0 0 10px 0;color:#666;">Use this to calculate the probability of a face appearing N or more times in the current three-die selection.</div>'
            ),
            exact_probability_output,
        ],
        layout=widgets.Layout(gap='12px'),
    )
)
render_exact_probability_summary()